In [1]:
import numpy as np
import pandas as pd
from utils.model_training import prepare_model_features, train_basic_cat_model, train_basic_LR_model
from sklearn.model_selection import train_test_split
import seaborn as sns

# Deep Learning Introduction
Its been a long road to this point and I refer you to the introduction in the first notebook for more context, but tl;dr this notebook is an attempt to model the use of violence by the police during stop and frisk interactions.

All of the data comes from the NYPD itself via its published yearly data on NYC's open data website.  The bulk of EDA is in the second notebook.  This notebook is dedicated to building upon the (imho) inadequate results from earlier modeling to determine if a neural net can generalize better than its less complex model peers.

For the sake of time and keeping things DRY, I will simply import the cleaned and prepared data, but the details are all included in the first and second notebooks.

In [2]:
df = pd.read_csv(
    "./data/processed/stop-and-frisk.csv",
    parse_dates=["STOP_FRISK_DATE"],  
)

X, y = prepare_model_features(df)
assert not X.isnull().any().any(), "Null values remain in features"

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


print("Train distribution:", y_train.value_counts(normalize=True), "\n")
print("Test distribution:", y_test.value_counts(normalize=True))

Train distribution: OFFICER_USED_FORCE
False    0.748507
True     0.251493
Name: proportion, dtype: float64 

Test distribution: OFFICER_USED_FORCE
False    0.748501
True     0.251499
Name: proportion, dtype: float64


In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np
import pandas as pd

class NYPDForcePredictor(nn.Module):
    def __init__(self, input_size, hidden_sizes=[128, 64, 32], dropout_rate=0.3):
        super(NYPDForcePredictor, self).__init__()
        
        # Build the network layers
        layers = []
        
        # Input layer
        layers.append(nn.Linear(input_size, hidden_sizes[0]))
        layers.append(nn.ReLU())
        layers.append(nn.BatchNorm1d(hidden_sizes[0]))
        layers.append(nn.Dropout(dropout_rate))
        
        # Hidden layers
        for i in range(len(hidden_sizes) - 1):
            layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i + 1]))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(hidden_sizes[i + 1]))
            layers.append(nn.Dropout(dropout_rate))
        
        # Output layer
        layers.append(nn.Linear(hidden_sizes[-1], 1))
        layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

class NYPDModelTrainer:
    def __init__(self, model, device='cpu'):
        self.model = model.to(device)
        self.device = device
        self.scaler = StandardScaler()
        
    def prepare_data(self, X, y, test_size=0.2, batch_size=512):
        """Prepare and scale the data for training"""
        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        
        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        # Convert to tensors
        X_train_tensor = torch.FloatTensor(X_train_scaled)
        X_test_tensor = torch.FloatTensor(X_test_scaled)
        y_train_tensor = torch.FloatTensor(y_train.values).unsqueeze(1)
        y_test_tensor = torch.FloatTensor(y_test.values).unsqueeze(1)
        
        # Create data loaders
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        
        return train_loader, test_loader, X_test_tensor, y_test_tensor
    
    def train(self, train_loader, test_loader, epochs=100, lr=0.001, weight_decay=1e-5, class_weighting=3.0):
        """Train the model with class weighting for imbalanced data"""
        # Calculate class weights (since force usage is 25% minority)
        pos_weight = torch.tensor([class_weighting]).to(self.device)  # 3.0 represents the 75/25 ratio underlying the class balances
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        
        # Use the model without final sigmoid for BCEWithLogitsLoss
        self.model.network = self.model.network[:-1]  # Remove sigmoid
        
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
        
        train_losses = []
        best_val_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_loss = 0.0
            
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
            
            # Validation phase
            self.model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch_X, batch_y in test_loader:
                    batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                    outputs = self.model(batch_X)
                    loss = criterion(outputs, batch_y)
                    val_loss += loss.item()
            
            train_loss /= len(train_loader)
            val_loss /= len(test_loader)
            train_losses.append(train_loss)
            
            scheduler.step(val_loss)
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                # Save best model
                torch.save(self.model.state_dict(), 'best_nypd_model.pth')
            else:
                patience_counter += 1
                
            if patience_counter >= 15:
                print(f"Early stopping at epoch {epoch}")
                break
            
            if epoch % 10 == 0:
                print(f'Epoch {epoch}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
        
        # Load best model
        self.model.load_state_dict(torch.load('best_nypd_model.pth'))
        
        # Add sigmoid back for predictions
        self.model.network = nn.Sequential(self.model.network, nn.Sigmoid())
        
        return train_losses
    
    def evaluate(self, X_test, y_test):
        """Evaluate the model and return metrics"""
        self.model.eval()
        
        with torch.no_grad():
            X_test = X_test.to(self.device)
            y_pred_proba = self.model(X_test).cpu().numpy()
            y_pred = (y_pred_proba > 0.5).astype(int)
        
        y_true = y_test.cpu().numpy()
        
        # Calculate metrics
        auc_score = roc_auc_score(y_true, y_pred_proba)
        
        print("Classification Report:")
        print(classification_report(y_true, y_pred))
        print(f"\nAUC Score: {auc_score:.4f}")
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_true, y_pred))
        
        return {
            'auc': auc_score,
            'predictions': y_pred,
            'probabilities': y_pred_proba
        }

# Usage Example:
def run_nypd_force_model(X, y, hidden_sizes=[128, 64, 32], dropout_rate=0.3, class_weighting = 3.0, lr=0.001):
    """
    Main function to run the NYPD force prediction model
    X, y should come from your prepare_model_features() function
    """
    
    # Initialize model
    input_size = X.shape[1]
    model = NYPDForcePredictor(
        input_size=input_size,
        hidden_sizes=hidden_sizes,
        dropout_rate=dropout_rate,
    )
    
    # Initialize trainer
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    trainer = NYPDModelTrainer(model, device)
    
    print(f"Using device: {device}")
    print(f"Input features: {input_size}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Prepare data
    train_loader, test_loader, X_test, y_test = trainer.prepare_data(X, y)
    
    # Train model
    print("\nStarting training...")
    train_losses = trainer.train(train_loader, test_loader, epochs=100, class_weighting=class_weighting, lr=lr)
    
    # Evaluate model
    print("\nEvaluating model...")
    results = trainer.evaluate(X_test, y_test)
    
    return trainer, results



# Model Training
I will train a variety of basic NN architectures varying across different hyperparameters for the hidden layer architecture as well as typical parameters (learning rate, dropout rate, etc.)

## Hyperparameter juicing is of little effect
Starting with a general foundation of hidden layer dimensions of [128,64,32], I tried doubling and halving the hidden dimensions to no general improvement of the model.  Similarly, increasing the dropout rate to .5/50% didn't improve the model, but did increase training time as expected.  

In [ ]:
trainer, results = run_nypd_force_model(X, y)

In [ ]:

trainer, results = run_nypd_force_model(X, y, [256, 128, 64])

Using device: cuda
Input features: 26
Model parameters: 49,025

Starting training...
Epoch 0: Train Loss: 1.0183, Val Loss: 0.9812
Epoch 10: Train Loss: 0.9769, Val Loss: 0.9756
Epoch 20: Train Loss: 0.9718, Val Loss: 0.9750
Early stopping at epoch 27

Evaluating model...
Classification Report:
              precision    recall  f1-score   support

         0.0       0.84      0.54      0.66      8241
         1.0       0.34      0.70      0.46      2769

    accuracy                           0.58     11010
   macro avg       0.59      0.62      0.56     11010
weighted avg       0.72      0.58      0.61     11010


AUC Score: 0.6638

Confusion Matrix:
[[4481 3760]
 [ 844 1925]]


In [18]:
trainer, results = run_nypd_force_model(X, y,[64, 32, 16])

Using device: cuda
Input features: 26
Model parameters: 4,577

Starting training...
Epoch 0: Train Loss: 1.0553, Val Loss: 0.9933
Epoch 10: Train Loss: 0.9855, Val Loss: 0.9810
Epoch 20: Train Loss: 0.9842, Val Loss: 0.9774
Epoch 30: Train Loss: 0.9793, Val Loss: 0.9777
Epoch 40: Train Loss: 0.9835, Val Loss: 0.9757
Epoch 50: Train Loss: 0.9777, Val Loss: 0.9759
Epoch 60: Train Loss: 0.9770, Val Loss: 0.9765
Epoch 70: Train Loss: 0.9744, Val Loss: 0.9754
Epoch 80: Train Loss: 0.9832, Val Loss: 0.9757
Early stopping at epoch 83

Evaluating model...
Classification Report:
              precision    recall  f1-score   support

         0.0       0.83      0.57      0.68      8241
         1.0       0.34      0.66      0.45      2769

    accuracy                           0.59     11010
   macro avg       0.59      0.62      0.56     11010
weighted avg       0.71      0.59      0.62     11010


AUC Score: 0.6620

Confusion Matrix:
[[4691 3550]
 [ 936 1833]]


In [12]:
trainer, results = run_nypd_force_model(X, y, dropout_rate=.5)


Using device: cuda
Input features: 26
Model parameters: 14,273

Starting training...
Epoch 0: Train Loss: 1.0626, Val Loss: 0.9916
Epoch 10: Train Loss: 0.9887, Val Loss: 0.9814
Epoch 20: Train Loss: 0.9853, Val Loss: 0.9780
Epoch 30: Train Loss: 0.9832, Val Loss: 0.9773
Epoch 40: Train Loss: 0.9833, Val Loss: 0.9767
Epoch 50: Train Loss: 0.9821, Val Loss: 0.9765
Epoch 60: Train Loss: 0.9782, Val Loss: 0.9762
Epoch 70: Train Loss: 0.9788, Val Loss: 0.9764
Early stopping at epoch 76

Evaluating model...
Classification Report:
              precision    recall  f1-score   support

         0.0       0.84      0.55      0.66      8241
         1.0       0.34      0.68      0.45      2769

    accuracy                           0.58     11010
   macro avg       0.59      0.62      0.56     11010
weighted avg       0.71      0.58      0.61     11010


AUC Score: 0.6613

Confusion Matrix:
[[4503 3738]
 [ 876 1893]]


In [13]:
trainer, results = run_nypd_force_model(X, y, class_weighting=2.0)


Using device: cuda
Input features: 26
Model parameters: 14,273

Starting training...
Epoch 0: Train Loss: 0.8601, Val Loss: 0.8203
Epoch 10: Train Loss: 0.7942, Val Loss: 0.7914
Epoch 20: Train Loss: 0.7955, Val Loss: 0.7905
Epoch 30: Train Loss: 0.7904, Val Loss: 0.7897
Epoch 40: Train Loss: 0.7859, Val Loss: 0.7901
Early stopping at epoch 41

Evaluating model...
Classification Report:
              precision    recall  f1-score   support

         0.0       0.79      0.80      0.80      8241
         1.0       0.39      0.37      0.38      2769

    accuracy                           0.70     11010
   macro avg       0.59      0.59      0.59     11010
weighted avg       0.69      0.70      0.69     11010


AUC Score: 0.6648

Confusion Matrix:
[[6624 1617]
 [1735 1034]]


In [14]:
trainer, results = run_nypd_force_model(X, y, class_weighting=1.0)


Using device: cuda
Input features: 26
Model parameters: 14,273

Starting training...
Epoch 0: Train Loss: 0.6953, Val Loss: 0.6484
Epoch 10: Train Loss: 0.5455, Val Loss: 0.5328
Epoch 20: Train Loss: 0.5339, Val Loss: 0.5326
Epoch 30: Train Loss: 0.5367, Val Loss: 0.5319
Epoch 40: Train Loss: 0.5335, Val Loss: 0.5317
Epoch 50: Train Loss: 0.5337, Val Loss: 0.5313
Early stopping at epoch 53

Evaluating model...
Classification Report:
              precision    recall  f1-score   support

         0.0       0.75      0.99      0.85      8241
         1.0       0.45      0.02      0.04      2769

    accuracy                           0.75     11010
   macro avg       0.60      0.51      0.45     11010
weighted avg       0.68      0.75      0.65     11010


AUC Score: 0.6630

Confusion Matrix:
[[8169   72]
 [2709   60]]


In [ ]:
# Learning rate modifications...
trainer, results = run_nypd_force_model(X, y, lr=0.003)


Using device: cuda
Input features: 26
Model parameters: 14,273

Starting training...
Epoch 0: Train Loss: 1.0193, Val Loss: 0.9820
Epoch 10: Train Loss: 0.9863, Val Loss: 0.9755
Epoch 20: Train Loss: 0.9762, Val Loss: 0.9759
Early stopping at epoch 29

Evaluating model...
Classification Report:
              precision    recall  f1-score   support

         0.0       0.84      0.54      0.66      8241
         1.0       0.34      0.69      0.45      2769

    accuracy                           0.58     11010
   macro avg       0.59      0.62      0.56     11010
weighted avg       0.71      0.58      0.61     11010


AUC Score: 0.6632

Confusion Matrix:
[[4480 3761]
 [ 856 1913]]


In [ ]:
# 10x LR
trainer, results = run_nypd_force_model(X, y, lr=0.01)


Using device: cuda
Input features: 26
Model parameters: 14,273

Starting training...
Epoch 0: Train Loss: 1.0009, Val Loss: 0.9806
Epoch 10: Train Loss: 0.9811, Val Loss: 0.9755
Epoch 20: Train Loss: 0.9857, Val Loss: 0.9760
Epoch 30: Train Loss: 0.9933, Val Loss: 0.9788
Early stopping at epoch 40

Evaluating model...
Classification Report:
              precision    recall  f1-score   support

         0.0       0.84      0.55      0.67      8241
         1.0       0.34      0.68      0.45      2769

    accuracy                           0.59     11010
   macro avg       0.59      0.62      0.56     11010
weighted avg       0.71      0.59      0.61     11010


AUC Score: 0.6640

Confusion Matrix:
[[4561 3680]
 [ 879 1890]]


In [ ]:
# Let's try some combos...
trainer, results = run_nypd_force_model(X, y, dropout_rate=.5, class_weighting=1.0)


Using device: cuda
Input features: 26
Model parameters: 14,273

Starting training...
Epoch 0: Train Loss: 0.6642, Val Loss: 0.6031
Epoch 10: Train Loss: 0.5434, Val Loss: 0.5325
Epoch 20: Train Loss: 0.5405, Val Loss: 0.5333
Epoch 30: Train Loss: 0.5357, Val Loss: 0.5319
Epoch 40: Train Loss: 0.5381, Val Loss: 0.5329
Epoch 50: Train Loss: 0.5380, Val Loss: 0.5318
Early stopping at epoch 54

Evaluating model...
Classification Report:
              precision    recall  f1-score   support

         0.0       0.75      1.00      0.86      8241
         1.0       0.49      0.01      0.03      2769

    accuracy                           0.75     11010
   macro avg       0.62      0.50      0.44     11010
weighted avg       0.69      0.75      0.65     11010


AUC Score: 0.6617

Confusion Matrix:
[[8201   40]
 [2730   39]]


In [16]:
trainer, results = run_nypd_force_model(X, y,hidden_sizes=[256,128,64],  dropout_rate=.5, class_weighting=2.5)


Using device: cuda
Input features: 26
Model parameters: 49,025

Starting training...
Epoch 0: Train Loss: 0.9556, Val Loss: 0.8971
Epoch 10: Train Loss: 0.8951, Val Loss: 0.8903
Epoch 20: Train Loss: 0.8969, Val Loss: 0.8899
Epoch 30: Train Loss: 0.8901, Val Loss: 0.8892
Epoch 40: Train Loss: 0.8920, Val Loss: 0.8890
Epoch 50: Train Loss: 0.8984, Val Loss: 0.8898
Early stopping at epoch 57

Evaluating model...
Classification Report:
              precision    recall  f1-score   support

         0.0       0.81      0.67      0.73      8241
         1.0       0.35      0.55      0.43      2769

    accuracy                           0.64     11010
   macro avg       0.58      0.61      0.58     11010
weighted avg       0.70      0.64      0.66     11010


AUC Score: 0.6633

Confusion Matrix:
[[5486 2755]
 [1256 1513]]


In [22]:
trainer, results = run_nypd_force_model(X, y,hidden_sizes=[256,128,64],  dropout_rate=.5, class_weighting=2.5, lr=0.01)

Using device: cuda
Input features: 26
Model parameters: 49,025

Starting training...
Epoch 0: Train Loss: 0.9291, Val Loss: 0.8944
Epoch 10: Train Loss: 0.9035, Val Loss: 0.8952
Epoch 20: Train Loss: 0.8951, Val Loss: 0.8899
Epoch 30: Train Loss: 0.8998, Val Loss: 0.8896
Epoch 40: Train Loss: 0.8928, Val Loss: 0.8900
Early stopping at epoch 42

Evaluating model...
Classification Report:
              precision    recall  f1-score   support

         0.0       0.81      0.70      0.75      8241
         1.0       0.36      0.50      0.42      2769

    accuracy                           0.65     11010
   macro avg       0.59      0.60      0.59     11010
weighted avg       0.70      0.65      0.67     11010


AUC Score: 0.6632

Confusion Matrix:
[[5807 2434]
 [1375 1394]]


In [23]:
trainer, results = run_nypd_force_model(X, y,hidden_sizes=[256,128,64],  dropout_rate=.5, class_weighting=2.5, lr=0.05)

Using device: cuda
Input features: 26
Model parameters: 49,025

Starting training...
Epoch 0: Train Loss: 0.9285, Val Loss: 0.9090
Epoch 10: Train Loss: 0.9157, Val Loss: 0.9351
Epoch 20: Train Loss: 0.9356, Val Loss: 0.9187
Epoch 30: Train Loss: 0.9051, Val Loss: 0.8914
Epoch 40: Train Loss: 0.9083, Val Loss: 0.8911
Epoch 50: Train Loss: 0.9022, Val Loss: 0.8972
Epoch 60: Train Loss: 0.8985, Val Loss: 0.8909
Early stopping at epoch 67

Evaluating model...
Classification Report:
              precision    recall  f1-score   support

         0.0       0.81      0.69      0.74      8241
         1.0       0.36      0.52      0.42      2769

    accuracy                           0.64     11010
   macro avg       0.58      0.60      0.58     11010
weighted avg       0.70      0.64      0.66     11010


AUC Score: 0.6619

Confusion Matrix:
[[5668 2573]
 [1340 1429]]


### Juicing moderately works
Tl;dr, increasing the hidden dimensions, increasing the dropout rate to 50%, decreasing the class imbalance weighting from 3.0 -> 2.5, and a 50x increase in learning rate from 0.001 -> 0.05 improved overall accuracy ~58% -> 64%. 

All of these results aren't markedly better than the linear supervised models that I initially trained, with top accuracy reaching ~70%, though both models only achieve this accuracy by overfitting the majority class. 

## Standing on the shoulders of giants (i.e. myself)
Though I didn't get fully improved results via unsupervised learning, I did pull out a new feature which did help model performance.  I will pull that in here.

In [ ]:
# pca_df = df.copy()
# pca_X, pca_y = prepare_model_features(pca_df)
# pca_X['Cluster'] = kmeans.labels_  

# clustered_X_train, clustered_X_test, clustered_y_train, clustered_y_test = train_test_split(pca_X, pca_y, test_size=0.2, random_state=42, stratify=y)


50.0

# Results + Conclusions
"Modelling is hard" - Zoolander 

Less flippantly, I really expected neural net architectures to generalize much better to be honest.  The overall model performance mirrored supervised models at best, but with far more computation.  

The confirmed benefits is that as neural nets are better equipped to handle higher dimensional data, I was able to confirm that the earlier models were not bottlenecked by the inclusion of too many features.

Regardless, I will continue this research until I get to a place where I can state some positive results.